In [ ]:
%%time
!pip install "stable-baselines3>=2.0.0" gymnasium>=0.29.0 "torch>=2.0.0" torchvision torchaudio shimmy swig -q
!pip install autorom[accept-rom-license] -q  # Gym wrappers
!apt update -qq &amp;&amp; apt install -y swig cmake libgl1-mesa-glx libglib2.0-0 -qq

# Verify installation
import subprocess, sys
try:
    import stable_baselines3 as sb3
    print(f"✅ Stable Baselines3: v{sb3.__version__}")
except:
    print("❌ Stable Baselines3 failed")

try:
    import gymnasium as gym
    print(f"✅ Gymnasium: v{gym.__version__}")
except:
    print("❌ Gymnasium failed")

try:
    import torch
    print(f"✅ PyTorch: v{torch.__version__}")
    print(f"✅ CUDA Available: {torch.cuda.is_available()}")
except:
    print("❌ PyTorch failed")

print("🎉 FIXED INSTALL COMPLETE!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 23.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/bin/bash: -c: line 1: syntax error near unexpected token `;&'
/bin/bash: -c: line 1: `apt update -qq &amp;&amp; apt install -y swig cmake libgl1-mesa-glx libglib2.0-0 -qq'
✅ Stable Baselines3: v2.7.1
✅ Gymnasium: v1.2.3
✅ PyTorch: v2.10.0+cu128
✅ CUDA Available: True
🎉 FIXED INSTALL COMPLETE!
CPU times: user 9.64 s, sys: 1.64 s, total: 11.3 s
Wall time: 29.2 s


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import torch
print(f"🚀 GPU Confirmed: {torch.cuda.get_device_name(0)}")
!nvidia-smi  # Memory check


🚀 GPU Confirmed: Tesla T4
Thu Mar 26 16:55:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
%%writefile rpg_combat_perfect.py
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class RPGCombatPerfect(gym.Env):
    def __init__(self, num_enemies=3):
        super().__init__()
        self.num_enemies = num_enemies
        self.current_step = 0

        # ✅ EXACTLY 30 dimensions - NO MISMATCH EVER
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(30,))
        self.action_space = spaces.MultiDiscrete([4] * num_enemies)  # 4 actions each

        self.player_pos = None
        self.player_health = None
        self.enemy_pos = None
        self.enemy_health = None

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)

        self.player_pos = np.array([0.0, 0.0])
        self.player_health = 100.0
        self.current_step = 0

        # 3 enemies at fixed positions
        self.enemy_pos = np.array([
            [3.0, 3.0], [-3.0, 3.0], [0.0, -4.0]
        ])
        self.enemy_health = np.array([100.0, 100.0, 100.0])

        obs = self._get_obs()
        return obs, {"player_health": self.player_health}

    def step(self, actions):
        self.current_step += 1

        reward = 0.0

        # Simple combat: enemies always try to damage player
        for i, action in enumerate(actions):
            dist = np.linalg.norm(self.enemy_pos[i] - self.player_pos)

            if action == 0 and dist < 2.0:  # Attack
                self.player_health -= 8.0
                reward += 10.0
            elif action == 1:  # Move closer
                direction = self.player_pos - self.enemy_pos[i]
                direction /= np.linalg.norm(direction) + 1e-6
                self.enemy_pos[i] += 0.2 * direction
                reward += 2.0
            elif action == 2:  # Dodge
                reward += 1.0
            else:  # Wait
                reward -= 1.0

        # Predictability penalty
        if len(np.unique(actions)) == 1:
            reward -= 15.0

        terminated = self.player_health <= 0
        truncated = self.current_step >= 300
        obs = self._get_obs()

        return obs, reward, terminated, truncated, {"player_health": self.player_health}

    def _get_obs(self):
        # ✅ ALWAYS 30 dimensions - mathematically guaranteed
        obs = np.zeros(30)

        # Player (5 dims)
        obs[0:2] = self.player_pos / 10.0
        obs[2] = self.player_health / 100.0
        obs[3] = self.current_step / 300.0
        obs[4] = 0.0  # padding

        # Enemies (3 enemies × 6 dims = 18 dims)
        for i in range(3):
            base = 5 + i * 6
            obs[base] = self.enemy_health[i] / 100.0
            obs[base+1:base+3] = self.enemy_pos[i] / 10.0
            obs[base+3] = np.linalg.norm(self.enemy_pos[i]) / 10.0
            obs[base+4:base+6] = 0.0  # padding

        # Team stats (7 dims)
        obs[23] = np.mean(self.enemy_health) / 100.0
        obs[24] = np.std(self.enemy_health) / 100.0
        obs[25] = np.mean(np.linalg.norm(self.enemy_pos, axis=1)) / 10.0
        obs[26] = len(np.unique(actions[-10:])) / 4.0 if hasattr(self, 'last_actions') else 1.0
        obs[27:30] = 0.0  # padding

        return obs.astype(np.float32)


Writing rpg_combat_perfect.py


In [ ]:
from rpg_combat_perfect import RPGCombatPerfect
env = RPGCombatPerfect(num_enemies=3)
obs, info = env.reset(seed=42)
print(f"✅ SHAPE PERFECT: {obs.shape}")
print(f"✅ Values OK: min={obs.min():.2f}, max={obs.max():.2f}")
print(f"✅ Health: {info['player_health']}")

action = env.action_space.sample()  # [0,1,2,3] for each enemy
obs, rew, term, trunc, info = env.step(action)
print(f"✅ STEP WORKS: reward={rew:.1f}, health={info['player_health']:.0f}")
print(f"✅ Still 30 dims: {obs.shape}")
env.close()


✅ SHAPE PERFECT: (30,)
✅ Values OK: min=-0.40, max=1.00
✅ Health: 100.0
✅ STEP WORKS: reward=-3.0, health=100
✅ Still 30 dims: (30,)


In [ ]:
%%time
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from rpg_combat_perfect import RPGCombatPerfect  # ✅ NEW IMPORT
import torch

print("🚀 T4 GPU detected, using CPU for stability")
env = make_vec_env(lambda: RPGCombatPerfect(), n_envs=2, seed=42)  # Reduced envs

model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=dict(net_arch=[256, 128, 64]),
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    device="cpu"
)

print("🎯 TRAINING STARTS NOW - 1M steps!")
model.learn(total_timesteps=1_000_000, progress_bar=True)
model.save("marl_rpg_perfect")
env.close()
print("✅ SAVED: marl_rpg_perfect.zip")


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Streaming output truncated to the last 5000 lines.
|    value_loss           | 1.23e+03    |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 53.3        |
|    ep_rew_mean          | 161         |
| time/                   |             |
|    fps                  | 534         |
|    iterations           | 4           |
|    time_elapsed         | 30          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.009046994 |
|    clip_fraction        | 0.0978      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.1        |
|    explained_variance   | 2.09e-06    |
|    learning_rate        | 0.0003      |
|    loss                 | 410         |
|    n_updates            | 30          |
|    policy_gradient_loss | -0.00806    |
|    value_loss           | 1.11e+03    |
-------------------------

✅ SAVED: marl_rpg_perfect.zip
CPU times: user 33min 46s, sys: 14 s, total: 34min
Wall time: 34min 49s


In [ ]:
%%time
from rpg_combat_perfect import RPGCombatPerfect
import numpy as np

print("🏃‍♂️ FAST Scripted Baseline (200 episodes = 2 min)")

def scripted_policy(obs):
    return np.array([0, 0, 0])  # Attack always

env = RPGCombatPerfect()
script_wins = 0
n_episodes = 200  # REDUCED from 1000

for i in range(n_episodes):
    obs, _ = env.reset()
    step_count = 0
    while not done and step_count < 350:  # SAFETY LIMIT
        action = scripted_policy(obs)
        obs, _, done, _, info = env.step(action)
        step_count += 1
    if info["player_health"] <= 0:
        script_wins += 1

script_win_rate = script_wins / n_episodes * 100
print(f"📊 SCRIPTED: {script_win_rate:.1f}% (Target: 35%)")
env.close()


🏃‍♂️ FAST Scripted Baseline (200 episodes = 2 min)
📊 SCRIPTED: 0.0% (Target: 35%)
CPU times: user 8.28 s, sys: 35 ms, total: 8.32 s
Wall time: 8.54 s


In [ ]:
%%time
from stable_baselines3 import PPO
from rpg_combat_perfect import RPGCombatPerfect
import numpy as np

print("🎯 YOUR MARL RESULTS (1000 episodes = 3-4 min)")

model = PPO.load("marl_rpg_perfect")
env = RPGCombatPerfect()
marl_wins = 0
n_episodes = 1000

for i in range(n_episodes):
    if i % 200 == 0:
        print(f"MARL Progress: {i}/{n_episodes}")
    obs, _ = env.reset()
    done = False
    step_count = 0
    while not done and step_count < 400:  # SAFETY
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
        step_count += 1
    if info["player_health"] <= 0:
        marl_wins += 1

marl_win_rate = marl_wins / n_episodes * 100
print(f"\n🎉 PAPER RESULTS:")
print(f"🚀 MARL Win Rate:    {marl_win_rate:.1f}% (Target: 55%)")
print(f"✅ TRAINING SUCCESS!" if marl_win_rate > 45 else "🔄 Needs more training")
env.close()


🎯 YOUR MARL RESULTS (1000 episodes = 3-4 min)
MARL Progress: 0/1000
MARL Progress: 200/1000
MARL Progress: 400/1000
MARL Progress: 600/1000
MARL Progress: 800/1000

🎉 PAPER RESULTS:
🚀 MARL Win Rate:    0.0% (Target: 55%)
🔄 Needs more training
CPU times: user 11min 11s, sys: 824 ms, total: 11min 11s
Wall time: 11min 15s
